# 034-structure-blind-rule: system run

# Imports

In [ ]:
import time
from dataclasses import asdict

import matplotlib.pyplot as plt
import pandas as pd

from uavloc.datasets import dataset_ref
from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.filtering.belief import BlindRule
from uavloc.simulation.arms import FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.sequence import all_fixes, flight_metrics, load_flight_set, per_flight_table, run_flights
from uavloc.track import REPO_ROOT, experiment_dir, load_runs, log_run, set_seed

# Params

In [ ]:
EXP = "034-structure-blind-rule"
REPLICATE = "A"                   # A: 027.12 models | B: 033 control | C: 033 noise
REGISTERED_RUN = "034.1"          # no rule 034.1-3, one-class 034.4-6, no-anchors 034.7-9 (A, B, C)
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
SPLIT = "test"
MAX_FLIGHTS = None                # first N flights only, for dry runs
INITIAL_ERROR_M = 0.0             # normal start
SIGMA0_M = 1.0
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"   # relative to src/
SEG_ARCHITECTURE = "lraspp"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
CLASS_WEIGHTS = [0.0, 1.0, 1.0, 0.5, 0.0, 0.5]
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
FILTER = "particles"
N_PARTICLES = 2000
K_MAX_HYPOTHESES = 5
PEAKS_PER_HYPOTHESIS = 3
PEAK_NMS_M = 10.0
MERGE_DISTANCE_M = 5.0
CHILD_FLOOR_SIGMA_M = 1.0
WINDOW_SIGMAS = 3.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
Q_POS_M_PER_SQRT_S = 2.0          # fixed, as for 027.12
TEMPERATURE = 0.5                 # fixed: every 027/033 arm chose 0.5; not retuned, so the rule is the only change
HEAD_PATH = "experiments/023-learned-correlation-head/artifacts/head-023.3.pt"   # relative to src/
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
HEAD_CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                       "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                       "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                       "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]   # villages no head trained on
BLIND_RULE = None                 # None | "one-class" | "no-anchors"
BLIND_THRESHOLD = None            # one-class: 0.95; no-anchors: 0.02
LOG_RUN = True
POLL_S = 30
NOTE = ""


In [ ]:
if not isinstance(REGISTERED_RUN, str):   # papermill -p reads 033.1 as the float 33.1; pass -y "REGISTERED_RUN: '033.1'"
    raise TypeError(f"REGISTERED_RUN must be a string, got {REGISTERED_RUN!r}")
set_seed(SEED)

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
specs = specs[:MAX_FLIGHTS]
len(specs)

In [ ]:
EXP_DIR = experiment_dir(EXP)
ARTIFACTS = EXP_DIR / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

# Model

In [ ]:
CORRELATOR = {
    "method": "segmentation", "seg_architecture": SEG_ARCHITECTURE,
    "seg_checkpoint_drone": SEG_CHECKPOINT, "seg_checkpoint_sat": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M, "matching": "weighted-ncc",
    "class_weights": CLASS_WEIGHTS, "window": "rect", "search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
}
HEAD_FILE = HEAD_PATH
HEAD = HeadSpec(REPO_ROOT / HEAD_FILE, tuple(HEAD_ENCODER_CHANNELS), HEAD_REFINE_CHANNELS)
HEAD.checkpoint.exists()

In [ ]:
BLIND = None if BLIND_RULE is None else BlindRule(BLIND_RULE, BLIND_THRESHOLD)
BLIND

In [ ]:
FILTER_PARAMS = FilterParams(TEMPERATURE, Q_POS_M_PER_SQRT_S, N_PARTICLES, K_MAX_HYPOTHESES, PEAKS_PER_HYPOTHESIS,
                             PEAK_NMS_M, MERGE_DISTANCE_M, CHILD_FLOOR_SIGMA_M, WINDOW_SIGMAS, WINDOW_CAP_HALF_M,
                             WINDOW_MARGIN_M, blind_rule=BLIND)

In [ ]:
def correlator_for(map_raster):
    return build_correlator(map_raster, CORRELATOR, DEVICE, HEAD)

# Run

In [ ]:
outcomes = run_flights(set_config, specs, INITIAL_ERROR_M, SIGMA0_M, correlator_for,
                       {FILTER: estimator_factory(FILTER, FILTER_PARAMS)})[FILTER]
len(outcomes)

In [ ]:
table = per_flight_table(outcomes)

In [ ]:
clean = [o for o in outcomes if o.spec.village in HEAD_CLEAN_VILLAGES]
flight_scores = {**flight_metrics(outcomes), **({f"head_clean_{k}": v for k, v in flight_metrics(clean).items()} if clean else {})}

In [ ]:
metrics = {**flight_scores, "flights_over_20m": int((table["end_error_m"] > 20.0).sum())}
pd.Series(metrics)

# Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_end_error_cdf({f"{REPLICATE} {BLIND_RULE}": table["end_error_m"]}, ax)
ax.set_title(f"034 {REGISTERED_RUN} replicate {REPLICATE}, rule {BLIND_RULE}, normal start, {SPLIT} flights ({len(specs)})")
fig.tight_layout()

# Log

In [ ]:
RUN_CONFIG = {"registered_run": REGISTERED_RUN, "arm": "particles", "replicate": REPLICATE,
              "blind_rule": None if BLIND is None else {"kind": BLIND.kind, "threshold": BLIND.threshold},
              "scenario": "normal", "split": SPLIT, "route_set": dataset_ref(ROUTE_SET), "initial_error_m": INITIAL_ERROR_M,
              "sigma0_m": SIGMA0_M, "ins": asdict(set_config.ins), "correlator": CORRELATOR, "head": HEAD_FILE,
              "filter": asdict(FILTER_PARAMS), "max_flights": MAX_FLIGHTS}

In [ ]:
def wait_for_turn() -> None:
    """Ids come from runs.jsonl order, so block until the registered predecessors are logged."""
    previous = int(REGISTERED_RUN.split(".")[1]) - 1
    while len(load_runs(EXP)) < previous:
        time.sleep(POLL_S)
    expected = f"{EXP.split('-')[0]}.{len(load_runs(EXP)) + 1}"
    if expected != REGISTERED_RUN:
        raise RuntimeError(f"run would log as {expected}, registered as {REGISTERED_RUN}")

In [ ]:
table.to_csv(ARTIFACTS / f"flights-{REGISTERED_RUN}.csv", index=False)
all_fixes(outcomes).to_csv(ARTIFACTS / f"fixes-{REGISTERED_RUN}.csv", index=False)

In [ ]:
if LOG_RUN:
    wait_for_turn()
    log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=f"{REGISTERED_RUN}: replicate {REPLICATE}, rule {BLIND_RULE}. {NOTE}",
            figures=[fig], seed=SEED, device=DEVICE)